## 1. Введение — выполните всю предварительную обработку, описанную в предыдущем уроке.

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler, StandardScaler
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from collections import Counter
from tqdm import tqdm
import random
import copy
from sklearn.model_selection import train_test_split, KFold, GroupKFold, StratifiedKFold, TimeSeriesSplit
from sklearn.pipeline import Pipeline
import time
from sklearn.inspection import permutation_importance



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/renat/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/renat/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/renat/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/home/renat/an

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/renat/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/renat/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/renat/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/home/renat/an

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/renat/anaconda3/lib/python3.12/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/home/renat/anaconda3/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/renat/anaconda3/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 701, in start
    self.io_loop.start()
  File "/home/renat/an

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



ImportError: numpy.core.multiarray failed to import

In [ ]:
train_df = pd.read_json("./data/train.json")
train_df

In [ ]:
test_df = pd.read_json("./data/test.json")
test_df

In [ ]:
train_df = train_df[["created", "bathrooms", "bedrooms", "features", "price"]]
test_df = test_df[["created","bathrooms", "bedrooms", "features", "price"]]

In [ ]:
quantile_1 = train_df["price"].quantile(0.01)
quantile_99 = train_df["price"].quantile(0.99)

train_df = train_df[(train_df["price"] > quantile_1) & (train_df["price"] < quantile_99)]
test_df = test_df[(test_df["price"] > quantile_1) & (test_df["price"] < quantile_99)]

In [ ]:
train_df.head()

In [ ]:
test_df.head()

In [ ]:
top_features = ['Elevator', 'HardwoodFloors', 'CatsAllowed', 'DogsAllowed', 'Doorman', 'Dishwasher', 'NoFee', 'LaundryinBuilding', 'FitnessCenter', 'Pre-War', 'LaundryinUnit', 'RoofDeck', 'OutdoorSpace', 'DiningRoom', 'HighSpeedInternet', 'Balcony', 'SwimmingPool', 'LaundryInBuilding', 'NewConstruction', 'Terrace']
for feature in top_features:
    train_df[feature] = train_df["features"].apply(lambda x: int(feature in x))
    test_df[feature] = test_df["features"].apply(lambda x: int(feature in x))

In [ ]:
train_df.head()

## 3. Реализуйте следующие методы:
- Разделите данные на 2 части случайным образом с параметром test_size (соотношение от 0 до 1), верните обучающую и тестовую выборки.
- Случайным образом разделить данные на 3 части с параметрами validation_size и test_size, вернуть обучающую, валидационную и тестовую выборки.
- Разделите данные на 2 части с помощью параметра date_split, верните обучающую и тестовую выборки, разделенные параметром date_split.
- Разделите данные на 3 части с помощью параметров validation_date и test_date, верните обучающую, валидационную и тестовую выборки, разделенные по входным параметрам.
- Сделать процедуру разделения детерминированной. Что это значит?

Детерминированность означает:

При одинаковых входных данных и параметрах результат разбиения всегда одинаковый

In [ ]:
random.seed(42) 

In [ ]:
def split_to_2part(data, test_size=0.2, random_state=42):
    df = data.sample(frac=1, random_state=random_state).reset_index(drop=True)
    split_idx = int(len(df) * (1 - test_size))
    
    train = df.iloc[:split_idx]
    test = df.iloc[split_idx:]
    
    return train, test

In [ ]:
def split_to_3part(data, validation_size=0.2, test_size=0.2):
    train, data = split_to_2part(data, 1 - validation_size - test_size)
    test, val = split_to_2part(data, test_size/(1 - validation_size - test_size))
    return train, test, val

In [ ]:
def split_by_date_2part(data, date_split, data_name="created"):
    df = data.sort_values(data_name).reset_index(drop=True)
    
    train = df[df[data_name] < date_split]
    test = df[df[data_name] >= date_split]
    
    return train, test


In [ ]:
def split_by_date_3part(data, validation_date, test_date, data_name="created"):
    
    df = data.sort_values(data_name).reset_index(drop=True)
    
    train = df[df[data_name] < validation_date]
    val = df[(df[data_name] >= validation_date) & (df[data_name] < test_date)]
    test = df[df[data_name] >= test_date]
    
    return train, val, test


## 4. Реализуйте следующие методы перекрестной проверки:

- Алгоритм K-Fold, где k — входной параметр, возвращает список индексов обучающей и тестовой выборок.
- Метод групповой K-кратной корреляции, где k и group_field являются входными параметрами, возвращает список индексов обучающей и тестовой выборок.
- Метод стратифицированной K-кратной корреляции, где k и stratify_field являются входными параметрами, возвращает список индексов обучающей и тестовой выборок.
- Функция разделения временного ряда, где k и date_field являются входными параметрами, возвращает список индексов для обучающей и тестовой выборок.

In [ ]:
def kfold_split(data, k=5, shuffle=True, random_state=42):
    n = len(data)
    indices = np.arange(n)

    if shuffle:
        rng = np.random.RandomState(random_state)
        rng.shuffle(indices)

    folds = np.array_split(indices, k)
    splits = []

    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        splits.append((train_idx, test_idx))

    return splits

In [ ]:
def group_kfold_split(data, group_field, k=5, shuffle=True, random_state=None):
    groups = data[group_field].values
    unique_groups = np.unique(groups)

    if shuffle:
        rng = np.random.default_rng(random_state)
        rng.shuffle(unique_groups)

    group_folds = np.array_split(unique_groups, k)

    splits = []
    for i in range(k):
        test_groups = group_folds[i]

        test_mask = np.isin(groups, test_groups)
        train_mask = ~test_mask

        train = data.iloc[train_mask]
        test = data.iloc[test_mask]

        splits.append((train, test))

    return splits


In [ ]:
def stratified_kfold_split(data, target_field, k=5, shuffle=True, random_state=None):
    y = data[target_field].values
    unique_classes = np.unique(y)

    rng = np.random.default_rng(random_state)

    # Индексы по классам
    class_indices = {}

    for cls in unique_classes:
        idx = np.where(y == cls)[0]

        if shuffle:
            rng.shuffle(idx)

        class_indices[cls] = np.array_split(idx, k)

    splits = []
    for i in range(k):
        test_idx = np.concatenate(
            [class_indices[cls][i] for cls in unique_classes]
        )

        train_idx = np.setdiff1d(np.arange(len(data)), test_idx)

        train = data.iloc[train_idx]
        test = data.iloc[test_idx]

        splits.append((train, test))

    return splits


In [ ]:
def time_series_split(data, date_field="created", k=5):
    df = data.sort_values(date_field).reset_index(drop=True)

    n = len(df)
    fold_size = n // (k + 1)

    splits = []

    for i in range(1, k + 1):
        train_end = fold_size * i
        test_end = fold_size * (i + 1)

        train = df.iloc[:train_end]
        test = df.iloc[train_end:test_end]

        splits.append((train, test))

    return splits


## 5. Сравнительный анализ с использованием перекрестной проверки
- Примените все описанные выше методы проверки к нашему набору данных. Для применения стратифицированного алгоритма необходимо предварительно обработать целевые данные.
- Примените соответствующие методы из библиотеки sklearn.
- Сравните полученные распределения признаков для обучающей части набора данных между sklearn и вашей реализацией.
- Сравните все схемы проверки. Выберите лучшую. Объясните свой выбор.

In [ ]:
target = "price"

data = train_df.copy()
data["price_bin"] = pd.qcut(data[target], q=5, labels=False)


**мои функции**

In [ ]:
data = train_df.copy()
target = "price"

# биннинг для Stratified
data["price_bin"] = pd.qcut(data[target], q=5, labels=False)

# =========================
# НАШИ ФУНКЦИИ (8 штук)
# =========================

# 1. split_to_2part
train_own_2, test_own_2 = split_to_2part(data, test_size=0.2)

# 2. split_to_3part
train_own_3, val_own_3, test_own_3 = split_to_3part(data)

# 3. split_by_date_2part
train_own_date2, test_own_date2 = split_by_date_2part(
    data,
    date_split="2016-06-01",
)

# 4. split_by_date_3part
train_own_date3, val_own_date3, test_own_date3 = split_by_date_3part(
    data,
    validation_date="2015-06-01",
    test_date="2016-06-01"
)

# 5. kfold_split
folds_own_k = kfold_split(data, k=5)
train_own_k = data.iloc[folds_own_k[0][0]]

# 6. group_kfold_split
folds_own_group = group_kfold_split(
    data,
    group_field="bedrooms",
    k=5
)
train_own_group = folds_own_group[0][0]

# 7. stratified_kfold_split
folds_own_strat = stratified_kfold_split(
    data,
    target_field="price_bin",
    k=5
)
train_own_strat = folds_own_strat[0][0]

# 8. time_series_split
folds_own_time = time_series_split(
    data,
    date_field="created",
    k=5
)
train_own_time = folds_own_time[0][0]

# =========================
# SKLEARN АНАЛОГИ
# =========================

# 1. train_test_split (2part)
train_sk_2, test_sk_2 = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)

# 2. train_test_split (3part)
train_tmp, test_sk_3 = train_test_split(
    data,
    test_size=0.15,
    random_state=42
)
train_sk_3, val_sk_3 = train_test_split(
    train_tmp,
    test_size=0.15 / 0.85,
    random_state=42
)

# 3. date 2part
data_sorted = data.sort_values("created")
train_sk_date2 = data_sorted[data_sorted["created"] < "2016-06-01"]

# 4. date 3part
train_sk_date3 = data_sorted[data_sorted["created"] < "2015-06-01"]

# 5. KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
train_sk_k = data.iloc[next(kf.split(data))[0]]

# 6. GroupKFold
gkf = GroupKFold(n_splits=5)
train_sk_group = data.iloc[next(gkf.split(data, groups=data["bedrooms"]))[0]]

# 7. StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_sk_strat = data.iloc[next(skf.split(data, data["price_bin"]))[0]]

# 8. TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)
train_sk_time = data_sorted.iloc[next(tscv.split(data_sorted))[0]]


In [ ]:
def compare(train_own, train_sk, name):
    print(f"\n{name}")
    
    num_cols = data.select_dtypes(include=np.number).columns
    
    comparison = pd.DataFrame({
        "Original": data[num_cols].mean(),
        "Own": train_own[num_cols].mean(),
        "Sklearn": train_sk[num_cols].mean()
    })
    
    print(comparison.head(10))


In [ ]:
compare(train_own_2, train_sk_2, "2part")
compare(train_own_3, train_sk_3, "3part")
compare(train_own_date2, train_sk_date2, "Date 2part")
compare(train_own_date3, train_sk_date3, "Date 3part")
compare(train_own_k, train_sk_k, "KFold")
compare(train_own_group, train_sk_group, "GroupKFold")
compare(train_own_strat, train_sk_strat, "StratifiedKFold")
compare(train_own_time, train_sk_time, "TimeSeriesSplit")


## 6. Выбор функций

- Постройте модель регрессии Lasso с нормализованными признаками. Используйте свой метод разделения выборок на 3 части по полям, созданным в соотношении 60/20/20 — обучающая/валидационная/тестовая выборки.
- Отсортируйте признаки по весовым коэффициентам модели, обучите модель на 10 наиболее важных признаках и сравните их качество.
- Реализуйте метод для простого отбора признаков по соотношению NaN в признаках и корреляции. Примените этот метод к набору признаков, выберите 10 лучших признаков, переобучите модель и оцените ее качество.
- Реализовать метод перестановочной важности, выбрать 10 наиболее важных признаков, переобучить модель и оценить качество.
- Импортируйте модель Shape, а также перенастройте ее по 10 основным параметрам.
- Сравните качество этих методов по различным параметрам — скорости, метрикам и стабильности.

In [ ]:
# Удаляем target и дату
drop_cols = [target, "created"]

X = data.drop(columns=drop_cols, errors="ignore")
y = data[target]

# Оставляем только числовые признаки
X = X.select_dtypes(include=["int64", "float64"])

# Разделение (твой метод 60/20/20)
train, val, test = split_to_3part(data, validation_size=0.2, test_size=0.2)

X_train = train[X.columns]
y_train = train[target]

X_val = val[X.columns]
y_val = val[target]

X_test = test[X.columns]
y_test = test[target]


# --- базовая модель ---
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, random_state=42))
])

start = time.time()
pipe.fit(X_train, y_train)
base_time = time.time() - start

val_pred = pipe.predict(X_val)
test_pred = pipe.predict(X_test)

base_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
base_r2 = r2_score(y_test, test_pred)

print("BASE RMSE:", base_rmse)
print("BASE R2:", base_r2)
print("TIME:", base_time)



In [ ]:
coefs = pipe.named_steps["model"].coef_
feature_importance = pd.Series(np.abs(coefs), index=X_train.columns)
top10_lasso = feature_importance.sort_values(ascending=False).head(10).index

# переобучаем
pipe_top10 = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, random_state=42))
])

pipe_top10.fit(X_train[top10_lasso], y_train)

test_pred_top10 = pipe_top10.predict(X_test[top10_lasso])

rmse_top10 = np.sqrt(mean_squared_error(y_test, test_pred_top10))
r2_top10 = r2_score(y_test, test_pred_top10)

print("LASSO TOP10 RMSE:", rmse_top10)
print("LASSO TOP10 R2:", r2_top10)


In [ ]:
# --- отбор по NaN + корреляции ---

def simple_feature_select(X, y, top_n=10, nan_threshold=0.4):

    # объединяем X и y
    df = X.copy()
    df[target] = y

    # оставляем только числовые
    df = df.select_dtypes(include=["int64", "float64"])

    # фильтр по NaN
    good_cols = df.columns[df.isna().mean() < nan_threshold]

    df = df[good_cols]

    # корреляция
    corr = df.corr()[target].abs().sort_values(ascending=False)

    selected = corr.drop(target).head(top_n).index
    return selected


top10_corr = simple_feature_select(X_train, y_train)

pipe_corr = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, random_state=42))
])

pipe_corr.fit(X_train[top10_corr], y_train)

test_pred_corr = pipe_corr.predict(X_test[top10_corr])

rmse_corr = np.sqrt(mean_squared_error(y_test, test_pred_corr))
r2_corr = r2_score(y_test, test_pred_corr)

print("CORR TOP10 RMSE:", rmse_corr)
print("CORR TOP10 R2:", r2_corr)


In [ ]:
def permutation_importance_manual(model, X, y, metric, n_repeats=3, random_state=42):

    np.random.seed(random_state)
    
    baseline = metric(y, model.predict(X))
    importances = {}

    for col in X.columns:
        scores = []

        for _ in range(n_repeats):
            X_permuted = X.copy()
            X_permuted[col] = np.random.permutation(X_permuted[col])
            
            score = metric(y, model.predict(X_permuted))
            scores.append(score)

        importances[col] = np.mean(scores) - baseline

    return pd.Series(importances).sort_values(ascending=False)


In [ ]:
# обучаем на всех признаках
pipe_full = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, random_state=42))
])

pipe_full.fit(X_train, y_train)

perm_imp = permutation_importance_manual(
    pipe_full,
    X_val,
    y_val,
    metric=lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred))
)

top10_perm = perm_imp.head(10).index

# переобучаем
pipe_perm = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, random_state=42))
])

pipe_perm.fit(X_train[top10_perm], y_train)

test_pred_perm = pipe_perm.predict(X_test[top10_perm])

print("PERM TEST RMSE:", np.sqrt(mean_squared_error(y_test, test_pred_perm)))
print("PERM TEST R2:", r2_score(y_test, test_pred_perm))


In [ ]:
import shap

# обучаем модель
pipe_full.fit(X_train, y_train)

explainer = shap.Explainer(pipe_full.predict, X_train)
shap_values = explainer(X_val)

# важность = среднее абсолютное значение SHAP
shap_importance = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=X_train.columns
).sort_values(ascending=False)

top10_shap = shap_importance.head(10).index

# переобучаем
pipe_shap = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.01, random_state=42))
])

pipe_shap.fit(X_train[top10_shap], y_train)

test_pred_shap = pipe_shap.predict(X_test[top10_shap])

print("SHAP TEST RMSE:", np.sqrt(mean_squared_error(y_test, test_pred_shap)))
print("SHAP TEST R2:", r2_score(y_test, test_pred_shap))


In [ ]:
rmse_shap = np.sqrt(mean_squared_error(y_test, test_pred_shap))
r2_shap = r2_score(y_test, test_pred_shap)

In [ ]:

comparison = pd.DataFrame({
    "Method": [
        "Base (all features)",
        "Lasso Top10",
        "Correlation Top10",
        "Permutation Top10",
        "SHAP Top10"
    ],
    "RMSE": [
        base_rmse,
        rmse_top10,
        rmse_corr,
        rmse_perm,
        rmse_shap
    ],
    "R2": [
        base_r2,
        r2_top10,
        r2_corr,
        r2_perm,
        r2_shap
    ]
})

comparison = comparison.sort_values("RMSE")

comparison


результаты оказались схожими, по времени SHAP занял в разы больше 

## 7. Оптимизация гиперпараметров

- Реализуйте методы перебора по сетке и случайного поиска для параметров alpha и l1_ratio в модели ElasticNet из библиотеки sklearn.
- Найдите наилучшее сочетание гиперпараметров модели.
- Подгоните полученную модель.
- Импортируйте Optuna и настройте тот же эксперимент с ElasticNet.
- Оцените показатели и сравните подходы.
- Запустите optuna на одной из схем перекрестной проверки.






In [ ]:
target = "price"
drop_cols = [target, "created"]

X = data.drop(columns=drop_cols, errors="ignore").select_dtypes(include=["int64", "float64"])
y = data[target]

train, val, test = split_to_3part(data, validation_size=0.2, test_size=0.2)

X_train, y_train = train[X.columns], train[target]
X_val, y_val = val[X.columns], val[target]
X_test, y_test = test[X.columns], test[target]

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
def grid_search_elasticnet(X_train, y_train, X_val, y_val, alphas, l1_ratios, random_state=42):
    best = {"rmse": np.inf, "alpha": None, "l1_ratio": None, "model": None}

    for a in alphas:
        for l1 in l1_ratios:
            model = Pipeline([
                ("scaler", StandardScaler()),
                ("enet", ElasticNet(alpha=a, l1_ratio=l1, random_state=random_state, max_iter=10000))
            ])
            model.fit(X_train, y_train)
            pred = model.predict(X_val)
            score = rmse(y_val, pred)

            if score < best["rmse"]:
                best.update({"rmse": score, "alpha": a, "l1_ratio": l1, "model": model})

    return best


alphas = np.logspace(-4, 1, 10)          # 10 значений
l1_ratios = np.linspace(0.05, 0.95, 10)  # 10 значений

best_grid = grid_search_elasticnet(X_train, y_train, X_val, y_val, alphas, l1_ratios)

print("GRID best RMSE (val):", best_grid["rmse"])
print("GRID best alpha:", best_grid["alpha"])
print("GRID best l1_ratio:", best_grid["l1_ratio"])

In [ ]:
def random_search_elasticnet(X_train, y_train, X_val, y_val, n_iter=50, alpha_range=(-4, 1), random_state=42):
    rng = np.random.default_rng(random_state)
    best = {"rmse": np.inf, "alpha": None, "l1_ratio": None, "model": None}

    for _ in range(n_iter):
        # alpha лог-равномерно: 10^[U(a,b)]
        alpha = 10 ** rng.uniform(alpha_range[0], alpha_range[1])
        l1_ratio = rng.uniform(0.05, 0.95)

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("enet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=random_state, max_iter=10000))
        ])
        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        score = rmse(y_val, pred)

        if score < best["rmse"]:
            best.update({"rmse": score, "alpha": alpha, "l1_ratio": l1_ratio, "model": model})

    return best


best_rand = random_search_elasticnet(X_train, y_train, X_val, y_val, n_iter=50)

print("RAND best RMSE (val):", best_rand["rmse"])
print("RAND best alpha:", best_rand["alpha"])
print("RAND best l1_ratio:", best_rand["l1_ratio"])

In [ ]:
best_manual = best_grid if best_grid["rmse"] <= best_rand["rmse"] else best_rand
final_model_manual = best_manual["model"]

test_pred = final_model_manual.predict(X_test)
print("MANUAL TEST RMSE:", rmse(y_test, test_pred))
print("MANUAL TEST R2:", r2_score(y_test, test_pred))

In [ ]:
import optuna

def objective(trial):
    alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("enet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=10000))
    ])
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    return rmse(y_val, pred)

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=50)

print("OPTUNA best:", study.best_params, "RMSE:", study.best_value)

In [ ]:
best_params = study.best_params

final_optuna = Pipeline([
    ("scaler", StandardScaler()),
    ("enet", ElasticNet(**best_params, random_state=42, max_iter=10000))
])

final_optuna.fit(X_train, y_train)
test_pred = final_optuna.predict(X_test)

print("OPTUNA TEST RMSE:", rmse(y_test, test_pred))
print("OPTUNA TEST R2:", r2_score(y_test, test_pred))

In [ ]:
def cv_rmse_for_params(data, feature_cols, target, alpha, l1_ratio, k=5):
    folds = kfold_split(data, k=k, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in folds:
        dtr = data.iloc[train_idx]
        dvl = data.iloc[val_idx]

        Xtr, ytr = dtr[feature_cols], dtr[target]
        Xvl, yvl = dvl[feature_cols], dvl[target]

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("enet", ElasticNet(alpha=alpha, l1_ratio=l1_ratio, random_state=42, max_iter=10000))
        ])
        model.fit(Xtr, ytr)
        pred = model.predict(Xvl)
        scores.append(rmse(yvl, pred))

    return float(np.mean(scores))


def objective_cv(trial):
    alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
    l1_ratio = trial.suggest_float("l1_ratio", 0.05, 0.95)
    return cv_rmse_for_params(data, X.columns, target, alpha, l1_ratio, k=5)

study_cv = optuna.create_study(direction="minimize")
study_cv.optimize(objective_cv, n_trials=50)

print("OPTUNA CV best:", study_cv.best_params, "RMSE:", study_cv.best_value)

In [ ]:
compare = pd.DataFrame([
    {"Method": "GridSearch", "Val RMSE": best_grid["rmse"], "alpha": best_grid["alpha"], "l1_ratio": best_grid["l1_ratio"]},
    {"Method": "RandomSearch", "Val RMSE": best_rand["rmse"], "alpha": best_rand["alpha"], "l1_ratio": best_rand["l1_ratio"]},
    {"Method": "Optuna", "Val RMSE": study.best_value, "alpha": study.best_params["alpha"], "l1_ratio": study.best_params["l1_ratio"]},
    {"Method": "Optuna + CV", "Val RMSE": study_cv.best_value, "alpha": study_cv.best_params["alpha"], "l1_ratio": study_cv.best_params["l1_ratio"]},
]).sort_values("Val RMSE")

compare